# Task 2
1、使用逻辑回归、SVM支持向量机对手写数字案例进行训练和预测

* 观察运行时间
* 观察准确率



2、对手写数字数据进行PCA降维，然后使用逻辑回归、SVM支持向量机对降维后的数据进行训练和预测

* 观察运行时间
* 观察准确率

## 导入包

In [ ]:
from __future__ import annotations

import asyncio
from typing import Any, Optional, Sequence, Union, List, Dict, Literal, Annotated

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from loguru import logger
from pathlib import Path

from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import BaseModel, Field

from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages


sns.set_theme(style="whitegrid")

preferred_cjk_fonts = [
    "Noto Sans CJK SC",
    "Noto Sans CJK TC",
    "Noto Sans CJK JP",
    "Microsoft YaHei",
    "SimHei",
    "Arial Unicode MS",
]
installed_fonts = {font.name for font in fm.fontManager.ttflist}
available_fonts = [name for name in preferred_cjk_fonts if name in installed_fonts]
plt.rcParams["font.sans-serif"] = available_fonts + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [44]:
class Settings(BaseSettings):
    data_path: Optional[Path] = Path("./data/inputs/digits.csv")
    openai_api_key: Optional[str]=None
    openai_base_url: Optional[str]=None
    openai_model: Optional[str]=None
    temperature: float = 0.0
    request_timeout: int = 30
    max_retries: int = 2
    enable_thinking: bool = True

    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        case_sensitive=False,
    )


cfg = Settings()

## 载入数据

In [ ]:
class Data:
    """数据加载、预处理与拆分。"""
    
    def __init__(self, file_path: Optional[Path | str]) -> None:
        self._data = self._load_data(file_path)
        self._X = self._data.drop(columns=["label"])
        self._y = self._data["label"]
    
    @property
    def X(self) -> pd.DataFrame:
        return self._X
    
    @property
    def y(self) -> pd.Series:
        return self._y
    
    def split_data(self, test_size: float = 0.2, random_state: int = 42) -> None:
        (
            self._X_train,
            self._X_test,
            self._y_train,
            self._y_test,
        ) = train_test_split(
            self._X, self._y,
            test_size=test_size,
            random_state=random_state,
            stratify=self._y,
        )
    
    @property
    def X_train(self) -> pd.DataFrame:
        return self._X_train
    
    @property
    def X_test(self) -> pd.DataFrame:
        return self._X_test
    
    @property
    def y_train(self) -> pd.Series:
        return self._y_train
    
    @property
    def y_test(self) -> pd.Series:
        return self._y_test

    def get_info(self):
        return self._data.info()

    def get_describe(self):
        return self._data.describe()

    def get_head(self, n: int = 5):
        return self._data.head(n)
    
    def _load_data(self, data_path: Optional[Union[Path, str]]) -> pd.DataFrame:
        if data_path is None:
            raise ValueError("data_path cannot be None")
        
        path = Path(data_path)
        if not path.exists():
            raise FileNotFoundError(f"Data file not found: {path}")
        if not path.is_file():
            raise ValueError(f"Path is not a file: {path}")
        
        try:
            df = pd.read_csv(path, encoding="utf-8")
            logger.info(f"Successfully loaded data from {path}")
            return df
        except UnicodeDecodeError:
            logger.warning(f"UTF-8 failed for {path}, trying latin-1...")
            df = pd.read_csv(path, encoding="latin-1")
            logger.info(f"Loaded data from {path} using latin-1 encoding")
            return df

data = Data(cfg.data_path)
data.split_data()

## 建模

### 非降维

In [28]:
SolverType = Literal["lbfgs", "newton-cg", "newton-cholesky", "sag", "saga"]

In [ ]:
def logistic_regression(
    data: Data,
    Cs: Sequence[float],
    cv: Optional[int],
    solvers: Sequence[SolverType],
    random_state: int = 42,
    max_iter: int = 5000,
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """逻辑回归 + 交叉验证调参。"""
    best_score = float("-inf")
    best_params: Dict[str, Union[float, str, np.ndarray, None]] = {
        "solver": None, "score": best_score, "C": None, "coef": None, "intercept": None,
    }

    if not Cs or not solvers:
        return None

    for solver in solvers:
        try:
            lr = LogisticRegressionCV(
                Cs=Cs, cv=cv, solver=solver,
                random_state=random_state, max_iter=max_iter, scoring="accuracy",
            )
            lr.fit(data.X_train, data.y_train)
            current_score = lr.score(data.X_test, data.y_test)

            if current_score > best_score:
                best_score = current_score
                best_params.update({
                    "C": lr.C_[0] if hasattr(lr.C_, "__len__") else lr.C_,
                    "coef": lr.coef_.copy() if lr.coef_ is not None else None,
                    "intercept": lr.intercept_[0] if hasattr(lr.intercept_, "__len__") else lr.intercept_,
                    "score": best_score,
                    "solver": solver,
                })
        except Exception as e:
            logger.exception(f"Failed to evaluate solver {solver}: {e}")
            continue

    return best_params if best_score != float("-inf") else None

In [ ]:
def gridcv_svm(
    data: Data,
    Cs: Sequence[float],
    kernel: str | Sequence[str] = "rbf",
    gammas: Sequence[float] | str = "auto",
    cv: Optional[int] = 5,
    **kwargs,
) -> Optional[Dict[str, Union[float, str, None]]]:
    """SVM + GridSearchCV 调参。"""
    if not Cs:
        raise ValueError("Cs sequence cannot be empty")
    if cv is not None and cv <= 0:
        raise ValueError("cv must be a positive integer")
    if any(c <= 0 for c in Cs):
        raise ValueError("All C values must be positive")

    kernel_values = [kernel] if isinstance(kernel, str) else list(kernel)
    gamma_values = [gammas] if isinstance(gammas, str) else list(gammas)

    param_grid = {"C": list(Cs), "kernel": kernel_values, "gamma": gamma_values}

    model = GridSearchCV(
        SVC(random_state=42), param_grid=param_grid,
        cv=cv, n_jobs=-1, scoring="accuracy",
    )

    try:
        model.fit(data.X_train, data.y_train)
        best_params = model.best_params_.copy()
        best_params["score"] = model.best_score_
        return best_params
    except Exception as e:
        logger.exception(f"Error during grid search: {e}")
        return None

### PCA降维之后建模

In [ ]:
def pca_logistic_regression(
    data: Data,
    Cs: Sequence[float],
    cv: Optional[int],
    solvers: Sequence[SolverType],
    random_state: int = 42,
    max_iter: int = 5000,
    n_components: Union[float, int, str] = 0.95,
    **kwargs,
) -> Optional[Dict[str, Union[float, str, np.ndarray, None]]]:
    """PCA 降维 + 逻辑回归交叉验证。"""
    if not Cs or not solvers:
        return None

    best_score = float("-inf")
    best_params: Dict[str, Union[float, str, np.ndarray, None]] = {
        "solver": None, "score": best_score, "C": None, "coef": None, "intercept": None,
    }

    # PCA 只需拟合一次
    pca = PCA(n_components=n_components, random_state=random_state)
    X_train_pca = pca.fit_transform(data.X_train)
    X_test_pca = pca.transform(data.X_test)

    for solver in solvers:
        try:
            lr = LogisticRegressionCV(
                Cs=Cs, cv=cv or 5, solver=solver,
                random_state=random_state, max_iter=max_iter, scoring="accuracy",
            )
            lr.fit(X_train_pca, data.y_train)
            current_score = lr.score(X_test_pca, data.y_test)

            if current_score > best_score:
                best_score = current_score
                best_params.update({
                    "C": lr.C_[0] if hasattr(lr.C_, "__len__") else lr.C_,
                    "coef": lr.coef_.copy() if lr.coef_ is not None else None,
                    "intercept": lr.intercept_[0] if hasattr(lr.intercept_, "__len__") else lr.intercept_,
                    "score": best_score,
                    "solver": solver,
                })
        except Exception as e:
            logger.exception(f"Failed to evaluate solver '{solver}': {e}")
            continue

    return best_params if best_score != float("-inf") else None

In [ ]:
def pca_gridcv_svm(
    data: Data,
    Cs: Sequence[float],
    kernel: Union[str, Sequence[str]] = "rbf",
    gammas: Sequence[float] | str = "auto",
    cv: Optional[int] = 5,
    n_components: Union[float, int, str] = 0.95,
    **kwargs,
) -> Optional[Dict[str, Union[float, str, None]]]:
    """PCA 降维 + SVM GridSearchCV 调参。"""
    if not Cs:
        raise ValueError("Parameter 'Cs' must contain at least one value.")
    if cv is not None and cv <= 0:
        raise ValueError("Parameter 'cv' must be a positive integer.")
    if any(c <= 0 for c in Cs):
        raise ValueError("All values in 'Cs' must be positive.")

    kernel_list = [kernel] if isinstance(kernel, str) else list(kernel)
    gamma_list = [gammas] if isinstance(gammas, str) else list(gammas)

    pipeline = Pipeline([
        ("pca", PCA(n_components=n_components)),
        ("svm", SVC(random_state=42)),
    ])

    param_grid = {
        "svm__C": list(Cs),
        "svm__kernel": kernel_list,
        "svm__gamma": gamma_list,
    }

    model = GridSearchCV(
        pipeline, param_grid=param_grid,
        cv=cv, n_jobs=-1, scoring="accuracy",
    )

    try:
        model.fit(data.X_train, data.y_train)
        best_params = model.best_params_.copy()
        best_params["score"] = model.best_score_
        return best_params
    except Exception as e:
        logger.exception(f"Error during PCA + SVM grid search: {e}")
        return None

### 手动训练

In [ ]:
lr_solver:Sequence[SolverType]=["lbfgs"]
logger.debug(logistic_regression(data,Cs=[1,10,100],cv=5,solvers=lr_solver,max_iter=10000))

In [ ]:
logger.debug(pca_logistic_regression(data,Cs=[10,100],cv=5,solvers=lr_solver))

In [ ]:
logger.debug(gridcv_svm(data, Cs=[10, 100]))

In [ ]:
logger.debug(pca_gridcv_svm(data, Cs=[10, 100]))

## agent workflow

In [ ]:
class Action(BaseModel):
    action_type: Literal["get_logistic_regression", "get_svm"]
    arguments: dict = Field(description="执行动作所需的参数")

class Plan(BaseModel):
    plan_summary: str = Field(description="计划的简短摘要")
    actions: List[Action] = Field(description="需要执行的行动列表")

class State(BaseModel):
    user_query: str
    plan: Union[Plan, None] = None
    results: List[str] = Field(default_factory=list)
    messages: Annotated[list, add_messages]

### 工具类

In [ ]:
@tool
def get_logistic_regression(
    params: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    """选择逻辑回归训练器，根据 enable_pca 决定是否先做 PCA 降维。"""
    cs = params.get("Cs", [0.1, 1.0, 10.0])
    cv = params.get("cv", 5)
    solvers = params.get("solvers", ["lbfgs"])
    other_params = {k: v for k, v in params.items() if k not in ("Cs", "cv", "solvers", "enable_pca")}

    if params.get("enable_pca", True):
        return pca_logistic_regression(data, cs, cv, solvers, **other_params)
    return logistic_regression(data, cs, cv, solvers, **other_params)

In [ ]:
@tool
def get_svm(
    params: Dict[str, Any],
) -> Optional[Dict[str, Any]]:
    """选择 SVM 训练器，根据 enable_pca 决定是否先做 PCA 降维。"""
    cs = params.get("Cs", [0.1, 1.0, 10.0])
    cv = params.get("cv", 5)
    kernel = params.get("kernel", "rbf")
    gammas = params.get("gammas", "auto")
    enable_pca = params.get("enable_pca", True)
    other_params = {k: v for k, v in params.items() if k not in ("Cs", "cv", "kernel", "gammas", "enable_pca")}

    if enable_pca:
        return pca_gridcv_svm(data, Cs=cs, kernel=kernel, gammas=gammas, cv=cv, **other_params)
    return gridcv_svm(data, Cs=cs, kernel=kernel, gammas=gammas, cv=cv, **other_params)

In [ ]:
llm = ChatOpenAI(
        model=cfg.openai_model, 
        api_key=cfg.openai_api_key,
        base_url=cfg.openai_base_url,
        temperature=cfg.temperature,
        max_retries=cfg.max_retries,
        timeout=cfg.request_timeout,
        streaming=True,
        stream_usage=True,
        extra_body={
            "enable_thinking":cfg.enable_thinking
        },
    )
structured_llm = llm.with_structured_output(Plan)

In [ ]:
PLAN_PROMPT = """你是一个ML训练调度器。根据用户请求，输出一个JSON格式的行动计划。

可用工具（action_type 只能是以下之一）：
- get_logistic_regression: 逻辑回归训练，arguments 示例: {{"Cs": [1, 10, 100], "cv": 5, "solvers": ["lbfgs"], "enable_pca": false}}
- get_svm: SVM训练，arguments 示例: {{"Cs": [1, 10, 100], "cv": 5, "kernel": "rbf", "gammas": "auto", "enable_pca": false}}

enable_pca=true 表示先做PCA降维再训练，enable_pca=false 表示直接训练。
为了对比降维与非降维效果，每个算法应分别生成 enable_pca=true 和 enable_pca=false 两个行动。

请直接输出JSON，不要解释。plan_summary 用一句话概括。

用户请求: {query}"""

async def plan_node(state: State) -> dict:
    """规划节点：根据用户请求生成结构化计划。"""
    logger.info(f"开始规划，用户请求: {state.user_query}")
    try:
        plan: Plan = await structured_llm.ainvoke(
            PLAN_PROMPT.format(query=state.user_query)
        )
        logger.info(f"规划成功，摘要: {plan.plan_summary}")
        logger.debug(f"完整计划: {plan.model_dump_json(indent=2)}")
        return {"plan": plan}
    except Exception as e:
        logger.error(f"规划失败: {e}", exc_info=True)
        return {
            "plan": None,
            "messages": [AIMessage(content=f"规划步骤时出错: {str(e)}")]
        }

In [ ]:
tool_node = ToolNode([get_logistic_regression, get_svm])

async def execute_tools_node(state: State) -> dict:
    """执行节点：将 Plan 中的行动转为 ToolNode 调用并收集结果。"""
    if not state.plan or not state.plan.actions:
        return {"results": [], "messages": []}

    logger.info(f"准备执行 {len(state.plan.actions)} 个行动")

    tool_calls = [
        {
            "id": f"call_{i}",
            "name": action.action_type,
            "args": {"params": action.arguments},
            "type": "tool_call",
        }
        for i, action in enumerate(state.plan.actions)
    ]
    ai_message = AIMessage(content="", tool_calls=tool_calls)

    try:
        tool_results = await asyncio.wait_for(
            tool_node.ainvoke({"messages": [ai_message]}),
            timeout=120.0,
        )
    except asyncio.TimeoutError:
        logger.error("工具执行整体超时")
        return {"results": ["部分工具执行超时"]}
    except Exception as e:
        logger.error(f"工具执行异常: {e}", exc_info=True)
        return {"results": [f"工具执行失败: {e}"]}

    results = [msg.content for msg in tool_results["messages"]]
    logger.info(f"所有工具执行完毕，结果: {results}")
    return {"results": results}

In [65]:
async def respond_node(state: State) -> dict:
    """响应节点：根据原始请求、计划和执行结果生成自然语言回复"""
    logger.info("生成最终回复")

    # 如果没有工具执行结果，给出默认回复
    if not state.results:
        default_msg = "抱歉，没有找到可执行的操作。"
        logger.warning(default_msg)
        return {"messages": [AIMessage(content=default_msg)]}

    # 构造提示，让 LLM 整合信息
    context = f"用户问题：{state.user_query}\n"
    context += f"计划摘要：{state.plan.plan_summary if state.plan else '无'}\n"
    results_formatted = "\n".join(f"- {r}" for r in state.results)
    context += f"执行结果：\n{results_formatted}\n"
    context += "请根据以上信息，生成一段友好、清晰的回复给用户（使用中文）。"

    try:
        response = await llm.ainvoke(context)
        msg_content = response.content if hasattr(response, 'content') else str(response)
        logger.info(f"回复生成成功: {msg_content}")
    except Exception as e:
        logger.error(f"生成回复失败: {e}", exc_info=True)
        msg_content = "处理请求时出现错误，请稍后重试。"

    return {"messages": [AIMessage(content=msg_content)]}

In [66]:
def should_execute_tools(state: State) -> Literal["execute_tools", "respond"]:
    """路由函数：判断是否还有需要执行的工具"""
    if state.plan and state.plan.actions:
        return "execute_tools"
    return "respond"

def create_graph() -> StateGraph:
    """构建并编译 LangGraph 工作流"""
    workflow = StateGraph(State)

    # 添加节点
    workflow.add_node("planner", plan_node)
    workflow.add_node("executor", execute_tools_node)
    workflow.add_node("responder", respond_node)

    # 定义边
    workflow.add_edge(START, "planner")
    workflow.add_conditional_edges(
        "planner",
        should_execute_tools,
        {
            "execute_tools": "executor",
            "respond": "responder",
        },
    )
    workflow.add_edge("executor", "responder")
    workflow.add_edge("responder", END)

    return workflow.compile()

In [ ]:
async def main():
    logger.info("启动 LangGraph 结构化代理")

    if not cfg.openai_api_key:
        raise ValueError("OPENAI_API_KEY 未设置，请在环境变量或 .env 文件中配置")

    graph = create_graph()
    user_query = "请帮我找出最佳算法及其最佳参数和准确率"

    initial_state = State(user_query=user_query, messages=[])

    final_state = None
    async for state in graph.astream(initial_state, stream_mode="values"):
        final_state = state
        logger.debug("当前状态:{}", state)

    if final_state and final_state.get("messages"):
        print("\n🤖 最终回复：")
        print(final_state["messages"][-1].content)
    else:
        print("未生成回复。")

In [ ]:
await main()